# Is the PT signal PT? A bin-level contamination audit (notebook 104)

**The question.** Each PT unit sums the 2 µm Visium HD bins inside one segmented tubule cross-section. RNA that
diffuses across the polygon edge, and edge bins that straddle two structures, add the neighbours' RNA to a unit.
Neighbours differ along the PT (S3 runs beside thick ascending limbs and collecting ducts), so contamination could
make spurious gradients; in AKI, infiltrate and oedema could make spurious "PT" injury calls.

**The frozen protocol** is `docs/results/pt-contamination-audit.md` (commit 465e253).
- Q1 size and Q2 mechanism: the non-native fraction of each unit, its fall with depth into the unit and its link to
  contact with each neighbour class.
- Q3: the non-native fractions along the axis.
- Q4 and Q5: do notebook 90's gradients and notebook 91's AKI calls survive when units are rebuilt from bins ≥ 6 µm
  inside the edge, against size-matched random bins?
- Q6: each calling method's exposure to edge-enriched genes, against an expression-matched null.

**Sections.** 1 setup; 2 bins to units (cached); 3 references and markers; 4 Q1–Q3; 5 Q4; 6 Q5; 7 Q6; 8 recovery
checks and verdicts; 9 figures; 10 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import inspect
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
PKG = results_root / 'package_pseudospace'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T93 = results_root / 'pt_pathway_networks' / 'tables'
output = results_root / 'pt_contamination_audit'
tables, figures, cache_root = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '104.contamination.1'
SEED = 104
RUNS = {'aki': ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'], 'hvm': ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']}
SPECIES_OF = {'Ctrl1A2': 'mouse', 'Ctrl1A4': 'mouse', 'IR2A2': 'mouse', 'IR2A4': 'mouse', 'HUK1_COR1': 'human', 'HUK1_MED1': 'human'}
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
CORE, LADDER, EDGE_MAX, N_LAYERS = 3, (1, 2, 4, 5), 1, 6
CLASSES = ['AL', 'DCT', 'CNT_CD', 'Glomerulus', 'outside']          # non-native classes with a reference
CONTACT = ['PT', 'AL', 'DCT', 'CNT_CD', 'Glomerulus', 'outside', 'other']
MIN_CP10K, RATIO, OTHER_RATIO = 1.0, 20.0, 2.0
GRADED, SURVIVE, NATIVE_MAX, EDGE_ENRICHED = .5, .5, .05, 1.0
LO, HI, PSEUDO = .3, .7, .5
N_NULL = 1000
REQUIRED = [T90 / 'd_end_contrasts.csv.gz', T91 / 'i_genes.csv.gz', T93 / 'calls_aki_vs_control.csv',
            T93 / 'calls_human_vs_mouse.csv', T71 / 'universe_aki_vs_control.csv', T71 / 'universe_human_vs_mouse.csv']
for run in RUNS:
    REQUIRED += [PKG / run / 'pt_units.h5ad', PKG / 'runs' / run / '01_units' / 'manifest.json',
                 PKG / 'runs' / run / '03_integration' / 'pass1_labelled.h5ad',
                 *[PKG / 'runs' / run / '01_units' / f'{s}_bin_to_unit.parquet' for s in RUNS[run]]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run the package export, notebooks 71, 90, 91 and 93 first)')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display

from pseudospace import contamination as ct
from pseudospace.external_replication import matched_null
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.stage_cache import cached_payload

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#D55E00', 'IR2A4': '#E69F00',
              'HUK1_COR1': '#009E73', 'HUK1_MED1': '#7fcdbb'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')

### 1.1 · Exports, positions and polygon classes (not cached)

G2: the AKI export has 14,048 units in the common support.

In [ ]:
guard_log = []
EXPORT, PASS1, MANIFEST = {}, {}, {}
for run, specimens in RUNS.items():
    units = read_pt_units(PKG / run / 'pt_units.h5ad', run)
    obs = units.obs
    mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(),
                                       obs.segment_class)
    frame = pd.DataFrame({'specimen': obs['sample'].astype(str).to_numpy()[mask],
                          'polygon': obs.polygon_id.astype(str).to_numpy()[mask], 'x': position},
                         index=obs.index[mask].astype(str))
    var = units.var.copy()
    EXPORT[run] = {'units': frame, 'genes': pd.Index(units.var_names.astype(str)),
                   'human': var['human_symbol'].astype(str) if 'human_symbol' in var else None,
                   'counts': sparse.csr_matrix(units.X[mask])}
    del units
    p1 = ad.read_h5ad(PKG / 'runs' / run / '03_integration' / 'pass1_labelled.h5ad', backed='r')
    PASS1[run] = p1.obs['coarse_class'].astype(str).copy()
    PASS1[run].index = PASS1[run].index.astype(str)
    del p1
    MANIFEST[run] = json.loads((PKG / 'runs' / run / '01_units' / 'manifest.json').read_text())
    print(run, frame.groupby('specimen').size().to_dict())
assert len(EXPORT['aki']['units']) == 14048, 'G2: AKI common support'
guard_log.append('G2: AKI export 14,048 units in the common support; human vs mouse '
                 f"{len(EXPORT['hvm']['units']):,}")


def bin_folder(run, specimen):
    names = list(MANIFEST[run]['samples'])
    return Path(MANIFEST[run]['inputs'][2 * names.index(specimen)])

## 2 · Bins to units (cached per run and specimen)

For each specimen: the bins, their polygon and class, depth, the unit aggregates (full, core, the ladder, R1, R2),
pooled PT-bin profiles by depth layer, pooled reference profiles per class and the contact shares.
G1 checks that the full aggregates equal the export counts.

In [ ]:
def read_bins(folder):
    """Bins x genes CSR counts and barcodes, from the 10x .h5 when present, else from the MTX folder."""
    if (folder / 'filtered_feature_bc_matrix.h5').is_file():
        with h5py.File(folder / 'filtered_feature_bc_matrix.h5', 'r') as f:
            m = f['matrix']
            shape = tuple(m['shape'][:])
            X = sparse.csc_matrix((m['data'][:], m['indices'][:], m['indptr'][:]), shape=shape).T.tocsr()
            return X, m['barcodes'][:].astype(str)
    from scipy.io import mmread
    mtx = folder / 'filtered_feature_bc_matrix'
    X = sparse.csr_matrix(mmread(mtx / 'matrix.mtx.gz').T)
    barcodes = pd.read_csv(mtx / 'barcodes.tsv.gz', header=None).iloc[:, 0].astype(str).to_numpy()
    return X, barcodes


def process(run, specimen):
    X, barcodes = read_bins(bin_folder(run, specimen))
    u1 = ad.read_h5ad(PKG / 'runs' / run / '01_units' / f'{specimen}_units.h5ad', backed='r')
    names = pd.Index(u1.var_names.astype(str))
    assert len(names) == X.shape[1], 'unit object and bin matrix gene counts differ'
    del u1
    exp = EXPORT[run]
    if SPECIES_OF[specimen] == 'human':
        cols = names.get_indexer(exp['human'].to_numpy())
    else:
        cols = names.get_indexer(exp['genes'])
    assert (cols >= 0).all(), 'export genes missing from the bin matrix'
    X = X[:, cols].tocsr()
    rows, colsb = ct.bin_grid(barcodes)
    b2u = pd.read_parquet(PKG / 'runs' / run / '01_units' / f'{specimen}_bin_to_unit.parquet')
    polygons = pd.Index(b2u.polygon_id.astype(str).unique())
    where = pd.Index(barcodes).get_indexer(b2u.bin_id.astype(str))
    assert (where >= 0).all() and not b2u.bin_id.duplicated().any()
    label = np.full(len(barcodes), -1)
    label[where] = polygons.get_indexer(b2u.polygon_id.astype(str))
    klass = np.asarray(polygons.map(lambda p: PASS1[run].get(f'{specimen}_{p}', 'unlabelled')), dtype=object)
    depth = ct.label_depth(rows, colsb, label)
    units = exp['units'][exp['units'].specimen.eq(specimen)]
    pt_poly = polygons.get_indexer(units.polygon.to_numpy())
    assert (pt_poly >= 0).all(), 'export unit without bins'
    lut = np.full(len(polygons), -1)
    lut[pt_poly] = np.arange(len(pt_poly))
    unit_of_bin = np.where(label >= 0, lut[label.clip(0)], -1)
    in_pt = unit_of_bin >= 0
    n = len(pt_poly)
    x = units.x.to_numpy()
    full = ct.unit_aggregate(X, unit_of_bin, n)
    full_ref = exp['counts'][np.flatnonzero(exp['units'].specimen.eq(specimen).to_numpy())]
    out = {'g1_absdiff': np.array(abs(full - full_ref).sum())}
    masks = {'full': None, **{f'core{d}': depth >= d for d in (CORE, *LADDER)}}
    for name, seed in (('R1', SEED), ('R2', 1000 + SEED)):
        masks[name] = ct.size_matched_sample(unit_of_bin, depth >= CORE, np.random.default_rng([seed, RUNS[run].index(specimen)]))
    for name, keep in masks.items():
        agg = full if keep is None else ct.unit_aggregate(X, unit_of_bin, n, keep=keep)
        if name == f'core{CORE}':
            out['core'] = agg
        for part, rows_ in (('early', x <= LO), ('late', x >= HI), ('all', np.ones(n, bool))):
            out[f'pool|{name}|{part}'] = np.asarray(agg[rows_].sum(axis=0)).ravel()
    layer = np.minimum(np.floor(depth), N_LAYERS).astype(int)
    out['layers'] = np.vstack([np.asarray(X[in_pt & (layer == k)].sum(axis=0)).ravel() for k in range(N_LAYERS + 1)])
    out['layer_bins'] = np.array([int((in_pt & (layer == k)).sum()) for k in range(N_LAYERS + 1)])
    out['edge'] = np.asarray(X[in_pt & (depth <= EDGE_MAX)].sum(axis=0)).ravel()
    out['interior'] = np.asarray(X[in_pt & (depth >= CORE)].sum(axis=0)).ravel()
    bin_class = np.where(label >= 0, klass[label.clip(0)], 'outside')
    for k in ('AL', 'DCT', 'CNT_CD', 'Glomerulus'):
        out[f'ref|{k}'] = np.asarray(X[(bin_class == k) & (depth >= CORE)].sum(axis=0)).ravel()
    out['ref|outside'] = np.asarray(X[label < 0].sum(axis=0)).ravel()
    out['n_outside_bins'] = np.array(int((label < 0).sum()))
    contact_class = {i: (c if c in ('PT', 'AL', 'DCT', 'CNT_CD', 'Glomerulus') else 'other') for i, c in enumerate(klass)}
    share = ct.contact_share(rows, colsb, label, contact_class, pt_poly, CONTACT)
    out['contact'] = share.to_numpy(float)
    out['n_bins'] = np.bincount(unit_of_bin[unit_of_bin >= 0], minlength=n)
    out['n_core_bins'] = np.bincount(unit_of_bin[(unit_of_bin >= 0) & (depth >= CORE)], minlength=n)
    return out


def pack(d):
    flat = {}
    for k, v in d.items():
        if sparse.issparse(v):
            v = sparse.csr_matrix(v)
            flat.update({f'{k}|data': v.data.astype(np.float32), f'{k}|indices': v.indices, f'{k}|indptr': v.indptr,
                         f'{k}|shape': np.array(v.shape)})
        else:
            flat[k] = np.asarray(v)
    return flat


def unpack(flat):
    out = {}
    for k in {key.split('|data')[0] for key in flat if key.endswith('|data')}:
        out[k] = sparse.csr_matrix((flat[f'{k}|data'], flat[f'{k}|indices'], flat[f'{k}|indptr']),
                                   shape=tuple(flat[f'{k}|shape']))
    out.update({k: v for k, v in flat.items() if '|data' not in k and '|indices' not in k and '|indptr' not in k
                and '|shape' not in k})
    return out


CODE = inspect.getsource(process) + inspect.getsource(ct)
BINS = {}
t0 = time.perf_counter()
for run, specimens in RUNS.items():
    for spec in specimens:
        payload = cached_payload(f'bins_{run}_{spec}', lambda: pack(process(run, spec)), root=cache_root,
                                 params={'core': CORE, 'ladder': LADDER, 'edge': EDGE_MAX, 'layers': N_LAYERS,
                                         'seed': SEED, 'logic': NOTEBOOK_LOGIC_VERSION},
                                 inputs={'export': str(PKG / run / 'pt_units.h5ad'), 'specimen': spec}, code=CODE)
        BINS[run, spec] = unpack(payload)
        print(f'{run} {spec}: {BINS[run, spec]["core"].shape[0]:,} units, G1 |diff| {float(BINS[run, spec]["g1_absdiff"]):.0f} '
              f'({time.perf_counter() - t0:.0f} s)')
assert all(float(b['g1_absdiff']) == 0 for b in BINS.values()), 'G1: rebuilt units differ from the export'
guard_log.append('G1: summing each export PT unit\'s bins reproduces its export counts exactly (8 specimen runs)')
print('\n'.join('· ' + g for g in guard_log))

## 3 · References and markers (per species)

In [ ]:
def cp10k(v):
    v = np.asarray(v, float)
    return v / v.sum() * 1e4


REF, MARKERS = {}, {}
marker_rows = []
for run, specimens in RUNS.items():
    genes = EXPORT[run]['genes']
    for sp in sorted({SPECIES_OF[s] for s in specimens}):
        group = [s for s in specimens if SPECIES_OF[s] == sp]
        pt = pd.Series(cp10k(sum(BINS[run, s]['interior'] for s in group)), genes)
        refs = {k: pd.Series(cp10k(sum(BINS[run, s][f'ref|{k}'] for s in group)), genes) for k in CLASSES}
        REF[run, sp] = {'PT': pt, **refs}
        for k in CLASSES:
            others = [refs[o] for o in CLASSES if o != k]
            genes_k = ct.class_markers(refs[k], pt, min_cp10k=MIN_CP10K, ratio=RATIO, others=others, other_ratio=OTHER_RATIO)
            MARKERS[run, sp, k] = genes_k
            for g in genes_k:
                marker_rows.append({'run': run, 'species': sp, 'class': k, 'gene': g, 'ref CP10k': refs[k][g],
                                    'PT interior CP10k': pt[g]})
MARKER_TABLE = pd.DataFrame(marker_rows)
MARKER_TABLE.to_csv(tables / 'markers.csv', index=False)
display(MARKER_TABLE.groupby(['run', 'species', 'class']).size().unstack())
for (run, sp, k), g in MARKERS.items():
    if run == 'aki' or sp == 'human':
        top = MARKER_TABLE[(MARKER_TABLE.run == run) & (MARKER_TABLE.species == sp) & (MARKER_TABLE['class'] == k)]
        print(run, sp, k, ', '.join(top.sort_values('ref CP10k', ascending=False).gene.head(12)))

## 4 · Q1–Q3: size, mechanism and position of the non-native signal

In [ ]:
def share_on(matrix, idx):
    m = sparse.csr_matrix(matrix)
    tot = np.asarray(m.sum(axis=1)).ravel().astype(float)
    on = np.asarray(m[:, idx].sum(axis=1)).ravel().astype(float)
    return on / np.maximum(tot, 1)


unit_rows, layer_rows = [], []
for run, specimens in RUNS.items():
    genes = EXPORT[run]['genes']
    for spec in specimens:
        sp = SPECIES_OF[spec]
        b = BINS[run, spec]
        here = EXPORT[run]['units'].specimen.eq(spec).to_numpy()
        units = EXPORT[run]['units'][here]
        full = EXPORT[run]['counts'][np.flatnonzero(here)]
        frame = pd.DataFrame({'run': run, 'unit': units.index, 'specimen': spec, 'x': units.x.to_numpy(),
                              'bins': b['n_bins'], 'core bins': b['n_core_bins'],
                              'counts': np.asarray(full.sum(axis=1)).ravel(),
                              'core counts': np.asarray(b['core'].sum(axis=1)).ravel()})
        for k in CLASSES:
            idx = genes.get_indexer(MARKERS[run, sp, k])
            ref = REF[run, sp]
            s_pt = ref['PT'].iloc[idx].sum() / 1e4
            s_ref = ref[k].iloc[idx].sum() / 1e4
            frame[f'f {k}'] = ct.nonnative_fraction(share_on(full, idx), s_pt, s_ref)
            frame[f'f {k} core'] = ct.nonnative_fraction(share_on(b['core'], idx), s_pt, s_ref)
            lay = b['layers']
            for layer in range(N_LAYERS + 1):
                layer_rows.append({'run': run, 'specimen': spec, 'class': k, 'layer': layer,
                                   'bins': int(b['layer_bins'][layer]),
                                   'marker share': float(lay[layer, idx].sum() / max(lay[layer].sum(), 1)),
                                   'f': float(ct.nonnative_fraction(lay[layer, idx].sum() / max(lay[layer].sum(), 1), s_pt, s_ref))})
        frame['F'] = frame[[f'f {k}' for k in CLASSES]].sum(axis=1)
        frame['F core'] = frame[[f'f {k} core' for k in CLASSES]].sum(axis=1)
        for i, c in enumerate(CONTACT):
            frame[f'contact {c}'] = b['contact'][:, i]
        unit_rows.append(frame)
UNITS = pd.concat(unit_rows, ignore_index=True)
UNITS.to_csv(tables / 'units.csv', index=False)
LAYERS = pd.DataFrame(layer_rows)
LAYERS.to_csv(tables / 'depth_profile.csv', index=False)

Q1 = UNITS.groupby(['run', 'specimen']).agg(units=('unit', 'size'), **{'median F': ('F', 'median'),
                                                                      'share F ≥ 0.2': ('F', lambda v: float((v >= .2).mean())),
                                                                      'median F core': ('F core', 'median'),
                                                                      'core share of counts': ('core counts', 'sum')})
Q1['core share of counts'] = Q1['core share of counts'] / UNITS.groupby(['run', 'specimen'])['counts'].sum()
for k in CLASSES:
    Q1[f'median f {k}'] = UNITS.groupby(['run', 'specimen'])[f'f {k}'].median()
Q1['mean F (post hoc)'] = UNITS.groupby(['run', 'specimen'])['F'].mean()   # medians sit at units with no marker counts
display(Q1.round(3))

tot = LAYERS.groupby(['run', 'specimen', 'layer'])['marker share'].sum().unstack('layer')   # Σ_k s_k per layer
Q2_DEPTH = pd.DataFrame({'Σ s layer 0': tot[0], 'Σ s layers ≥ 3 (bin-weighted)': np.nan})
for (run, spec), row in tot.iterrows():
    w = LAYERS[(LAYERS.run == run) & (LAYERS.specimen == spec) & (LAYERS['class'] == CLASSES[0])].set_index('layer').bins
    deep = [l for l in range(CORE, N_LAYERS + 1)]
    Q2_DEPTH.loc[(run, spec), 'Σ s layers ≥ 3 (bin-weighted)'] = float(np.average(row[deep], weights=w.loc[deep]))
Q2_DEPTH['ratio'] = Q2_DEPTH['Σ s layer 0'] / Q2_DEPTH['Σ s layers ≥ 3 (bin-weighted)']
EDGE_SPILL = bool((Q2_DEPTH['ratio'] >= 2).all())
display(tot.round(4)); display(Q2_DEPTH.round(4))
print('Edge spillover holds (layer 0 ≥ 2 × layers ≥ 3 in every specimen):', EDGE_SPILL)

contact_rows = []
for (run, spec), g in UNITS.groupby(['run', 'specimen']):
    for k in CLASSES:
        rho = spearmanr(g[f'f {k}'], g[f'contact {k}']).statistic
        contact_rows.append({'run': run, 'specimen': spec, 'class': k, 'rho': rho})
Q2_CONTACT = pd.DataFrame(contact_rows).pivot_table(index=['run', 'specimen'], columns='class', values='rho')
CONTACT_HOLDS = (Q2_CONTACT > 0).all()
display(Q2_CONTACT.round(3))
print('Contact predicts contamination (rho > 0 in every specimen):', CONTACT_HOLDS.to_dict())

edges = np.linspace(0, 1, 11)
UNITS['xbin'] = np.clip(np.digitize(UNITS.x, edges[1:-1]), 0, 9)
ALONG = UNITS.groupby(['run', 'specimen', 'xbin'])[[f'f {k}' for k in CLASSES] + ['F']].mean()
Q3 = UNITS.assign(end=np.where(UNITS.x >= HI, 'late', np.where(UNITS.x <= LO, 'early', 'mid')))
Q3 = Q3[Q3.end != 'mid'].groupby(['run', 'specimen', 'end'])[[f'f {k}' for k in CLASSES] + ['F']].mean().unstack('end')
Q3_END = pd.concat({c: Q3[c]['late'] - Q3[c]['early'] for c in [f'f {k}' for k in CLASSES] + ['F']}, axis=1)
display(Q3_END.round(4))
aki_u = UNITS[UNITS.run == 'aki']
IR_OUTSIDE = float(aki_u[aki_u.specimen.isin(IRS)]['f outside'].mean() - aki_u[aki_u.specimen.isin(CONTROLS)]['f outside'].mean())
IR_F = float(aki_u[aki_u.specimen.isin(IRS)]['F'].mean() - aki_u[aki_u.specimen.isin(CONTROLS)]['F'].mean())
print(f'AKI: IR − control mean f_outside {IR_OUTSIDE:+.4f}; mean F {IR_F:+.4f}')

## 5 · Q4: do notebook 90's gradients survive in the core?

In [ ]:
END90 = pd.read_csv(T90 / 'd_end_contrasts.csv.gz').set_index('mouse')
END90.index = END90.index.astype(str)
genes_h = EXPORT['hvm']['genes']
AGGS = ['full', f'core{CORE}', 'R1', 'R2', *[f'core{d}' for d in LADDER]]
def pooled(run, spec, a, part):
    return ct.log_cp10k(BINS[run, spec][f'pool|{a}|{part}'][None, :], PSEUDO)


EHAT = {}
for spec in RUNS['hvm']:
    for a in AGGS:
        EHAT[spec, a] = pd.Series(pooled('hvm', spec, a, 'late') - pooled('hvm', spec, a, 'early'), genes_h)
G3 = {}
for spec in RUNS['hvm']:
    common = END90.index.intersection(genes_h)
    G3[spec] = float(spearmanr(EHAT[spec, 'full'][common], END90.loc[common, f'E {spec}']).statistic)
print('G3 Spearman(Ê full, 90 E):', {k: round(v, 3) for k, v in G3.items()})
guard_log.append('G3 (reported): Spearman of the pooled end contrast against 90\'s curve contrast: '
                 + ', '.join(f'{k} {v:.2f}' for k, v in G3.items()))


def survives(sign, full, agg):
    return (np.sign(agg) == sign) & (np.abs(agg) >= SURVIVE * np.abs(full))


q4_rows, Q4 = [], {}
for sp, pair in (('mouse', MOUSE), ('human', HUMAN)):
    e1, e2 = END90[f'E {pair[0]}'], END90[f'E {pair[1]}']
    graded = END90.index[(np.sign(e1) == np.sign(e2)) & (e1.abs() >= GRADED) & (e2.abs() >= GRADED)].intersection(genes_h)
    sign = np.sign(END90.loc[graded, f'E {pair[0]}'])
    ok = {a: pd.concat([survives(sign, EHAT[s, 'full'][graded], EHAT[s, a][graded]) for s in pair], axis=1).all(axis=1)
          for a in AGGS[1:]}
    sens = {d: ok['R1'] & ok['R2'] & ~ok[f'core{d}'] for d in (CORE, *LADDER)}
    frame = pd.DataFrame({'species': sp, 'gene': graded, 'E sign': sign.to_numpy(),
                          **{f'Ê {a} {s}': EHAT[s, a][graded].to_numpy() for s in pair for a in ('full', f'core{CORE}', 'R1', 'R2')},
                          **{f'survives {a}': ok[a].to_numpy() for a in AGGS[1:]},
                          **{f'sensitive core{d}': sens[d].to_numpy() for d in (CORE, *LADDER)}})
    q4_rows.append(frame)
    Q4[sp] = {'graded': len(graded), 'survive core': float(ok[f'core{CORE}'].mean()), 'survive R1': float(ok['R1'].mean()),
              'survive R2': float(ok['R2'].mean()), 'sensitive': int(sens[CORE].sum()),
              'sensitive share': float(sens[CORE].mean()),
              **{f'sensitive share core{d}': float(sens[d].mean()) for d in LADDER},
              'native': bool(sens[CORE].mean() <= NATIVE_MAX)}
GRAD = pd.concat(q4_rows, ignore_index=True)
GRAD.to_csv(tables / 'gradient_robustness.csv.gz', index=False)
display(pd.DataFrame(Q4).T)
for sp in ('mouse', 'human'):
    s = GRAD[(GRAD.species == sp) & GRAD[f'sensitive core{CORE}']]
    print(sp, 'contamination-sensitive graded genes:', ', '.join(s.gene.head(40)), '…' if len(s) > 40 else '')

## 6 · Q5: do notebook 91's AKI calls survive in the core?

In [ ]:
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
genes_a = EXPORT['aki']['genes']
LHAT, PHAT_E = {}, {}
for spec in RUNS['aki']:
    for a in AGGS:
        LHAT[spec, a] = pd.Series(pooled('aki', spec, a, 'all'), genes_a)
        PHAT_E[spec, a] = pd.Series(pooled('aki', spec, a, 'late') - pooled('aki', spec, a, 'early'), genes_a)
PAIRS = [(i, c) for i in IRS for c in CONTROLS]
q5_rows, Q5 = [], {}
for kind, flag, ref_col in (('level', 'level change', 'L'), ('location', 'location change', 'Δ late')):
    called = G91.index[G91[flag].astype(bool)].intersection(genes_a)
    sign = np.sign(G91.loc[called, ref_col])
    ok, est = {}, {}
    for a in AGGS:
        if kind == 'level':
            per_pair = np.vstack([(LHAT[i, a] - LHAT[c, a])[called].to_numpy() for i, c in PAIRS])
            est[a] = per_pair
        else:
            est[a] = (np.mean([PHAT_E[i, a][called] for i in IRS], axis=0)
                      - np.mean([PHAT_E[c, a][called] for c in CONTROLS], axis=0))
    for a in AGGS[1:]:
        if kind == 'level':
            ok[a] = (np.sign(est[a]) == sign.to_numpy()).all(axis=0) & (np.abs(est[a]).mean(axis=0) >= SURVIVE * np.abs(est['full']).mean(axis=0))
        else:
            ok[a] = (np.sign(est[a]) == sign.to_numpy()) & (np.abs(est[a]) >= SURVIVE * np.abs(est['full']))
    sens = {d: ok['R1'] & ok['R2'] & ~ok[f'core{d}'] for d in (CORE, *LADDER)}
    full_est = est['full'].mean(axis=0) if kind == 'level' else est['full']
    core_est = est[f'core{CORE}'].mean(axis=0) if kind == 'level' else est[f'core{CORE}']
    q5_rows.append(pd.DataFrame({'kind': kind, 'gene': called, 'sign': sign.to_numpy(), 'full': full_est, 'core': core_est,
                                 **{f'survives {a}': ok[a] for a in AGGS[1:]},
                                 **{f'sensitive core{d}': sens[d] for d in (CORE, *LADDER)}}))
    Q5[kind] = {'calls': len(called), 'survive core': float(ok[f'core{CORE}'].mean()), 'survive R1': float(ok['R1'].mean()),
                'survive R2': float(ok['R2'].mean()), 'sensitive': int(sens[CORE].sum()), 'sensitive share': float(sens[CORE].mean()),
                **{f'sensitive share core{d}': float(sens[d].mean()) for d in LADDER}, 'native': bool(sens[CORE].mean() <= NATIVE_MAX)}
AKI_ROB = pd.concat(q5_rows, ignore_index=True)
AKI_ROB.to_csv(tables / 'aki_robustness.csv.gz', index=False)
display(pd.DataFrame(Q5).T)
for kind in ('level', 'location'):
    s = AKI_ROB[(AKI_ROB.kind == kind) & AKI_ROB[f'sensitive core{CORE}']]
    print(kind, 'contamination-sensitive AKI calls:', ', '.join(s.gene.head(40)), '…' if len(s) > 40 else '')

## 7 · Q6: edge enrichment, G4 and each method's exposure

In [ ]:
EPS = {}
for run, specimens in RUNS.items():
    genes = EXPORT[run]['genes']
    for spec in specimens:
        b = BINS[run, spec]
        EPS[run, spec] = pd.Series(np.log2((b['edge'] + PSEUDO) / b['edge'].sum() * 1e4)
                                   - np.log2((b['interior'] + PSEUDO) / b['interior'].sum() * 1e4), genes)
EPS_TABLE = pd.concat({f'{run} {spec}': v for (run, spec), v in EPS.items()}, axis=1)
EPS_TABLE.to_csv(tables / 'edge_enrichment.csv.gz')
ENRICHED = {}
for run, specimens in RUNS.items():
    for sp in sorted({SPECIES_OF[s] for s in specimens}):
        group = [s for s in specimens if SPECIES_OF[s] == sp]
        ENRICHED[run, sp] = pd.concat([EPS[run, s] >= EDGE_ENRICHED for s in group], axis=1).all(axis=1)
g4_rows = []
for run, specimens in RUNS.items():
    for sp in sorted({SPECIES_OF[s] for s in specimens}):
        group = [s for s in specimens if SPECIES_OF[s] == sp]
        mean_eps = pd.concat([EPS[run, s] for s in group], axis=1).mean(axis=1)
        markers = pd.Index(sorted(set().union(*[set(MARKERS[run, sp, k]) for k in CLASSES])))
        top_pt = REF[run, sp]['PT'].sort_values(ascending=False).index[:50]
        g4_rows.append({'run': run, 'species': sp, 'markers': len(markers), 'median ε markers': float(mean_eps[markers].median()),
                        'median ε top-50 PT genes': float(mean_eps[top_pt].median()),
                        'edge-enriched genes': int(ENRICHED[run, sp].sum())})
G4 = pd.DataFrame(g4_rows)
G4_PASS = bool(((G4['median ε markers'] >= 1) & (G4['median ε top-50 PT genes'] <= .5)).all())
display(G4.round(3))
guard_log.append(f'G4: edge enrichment tracks non-native RNA (markers ≥ 1, top PT ≤ 0.5 in every run and species): {G4_PASS}')

exposure_rows = []
for run, contrast in (('aki', 'aki_vs_control'), ('hvm', 'human_vs_mouse')):
    universe = pd.read_csv(T71 / f'universe_{contrast}.csv').gene.astype(str)
    universe = pd.Index(universe).intersection(EXPORT[run]['genes'])
    if run == 'aki':
        hit = ENRICHED['aki', 'mouse'][universe]
    else:
        hit = ENRICHED['hvm', 'mouse'][universe] | ENRICHED['hvm', 'human'][universe]
    level = pd.concat([pd.Series(pooled(run, s, 'full', 'all'), EXPORT[run]['genes']) for s in RUNS[run]], axis=1).mean(axis=1)
    decile = pd.qcut(level[universe].rank(method='first'), 10, labels=False)
    calls = pd.read_csv(T93 / f'calls_{contrast}.csv')
    for method, g in calls.groupby('method'):
        drivers = pd.Index(sorted(set(','.join(g['driver gene list'].dropna().astype(str)).split(',')) - {''})).intersection(universe)
        if len(drivers) == 0:
            continue
        obs_hits = int(hit[drivers].sum())
        null = matched_null(hit.to_numpy(), decile.to_numpy(), decile[drivers].to_numpy(), n=N_NULL, seed=SEED)
        exposure_rows.append({'contrast': contrast, 'method': method, 'calls': len(g), 'driver genes': len(drivers),
                              'edge-enriched': obs_hits, 'exposure': obs_hits / len(drivers),
                              'null mean': float(null.mean() / len(drivers)), 'ratio': obs_hits / max(null.mean(), 1e-9),
                              'p (≥)': float((1 + (null >= obs_hits).sum()) / (N_NULL + 1)),
                              'flagged genes': ','.join(drivers[hit[drivers].to_numpy()][:30])})
EXPOSURE = pd.DataFrame(exposure_rows)
EXPOSURE.to_csv(tables / 'method_exposure.csv', index=False)
display(EXPOSURE.drop(columns='flagged genes').round(4))

## 8 · Recovery checks and verdicts

In [ ]:
rec = []
for g in ('Egf', 'Klk1', 'Muc1'):
    vals = [EPS['aki', s].get(g, np.nan) for s in CONTROLS]
    rec.append({'check': 'RC1 edge-enriched in the mouse controls', 'gene': g, 'value': ', '.join(f'{v:.2f}' for v in vals),
                'met': bool(all(v >= EDGE_ENRICHED for v in vals))})
for sp, run in (('mouse', 'aki'), ('human', 'hvm')):
    for g, k in (('Umod', 'AL'), ('Slc12a3', 'DCT'), ('Aqp2', 'CNT_CD'), ('Nphs2', 'Glomerulus')):
        present = g in EXPORT[run]['genes']
        rec.append({'check': f'RC2 marker of {k} ({sp})', 'gene': g, 'value': 'on panel' if present else 'not on panel',
                    'met': bool(g in set(MARKERS[run, sp, k])) if present else None})
for g in ('Alb', 'Ttr'):
    vals = [EPS['hvm', s].get(g, np.nan) for s in HUMAN]
    rec.append({'check': 'RC3 expected miss: human ambient genes not edge-enriched', 'gene': g,
                'value': ', '.join(f'{v:.2f}' for v in vals), 'met': bool(not all(v >= EDGE_ENRICHED for v in vals))})
REC = pd.DataFrame(rec)
REC.to_csv(tables / 'recovery_checks.csv', index=False)
display(REC)

VERDICTS = pd.DataFrame([
    ('Q1 size', 'median F per specimen ' + ', '.join(f'{r}/{s} {v:.3f}' for (r, s), v in Q1['median F'].items())),
    ('Q2 edge spillover (layer 0 ≥ 2 × core)', str(EDGE_SPILL)),
    ('Q2 contact predicts contamination', json.dumps({k: bool(v) for k, v in CONTACT_HOLDS.items()})),
    ('Q3 AKI IR − control f_outside, F', f'{IR_OUTSIDE:+.4f}, {IR_F:+.4f}'),
    ('Q4 gradients native (≤ 5% sensitive)', json.dumps({sp: Q4[sp]['native'] for sp in Q4})),
    ('Q5 AKI calls native (≤ 5% sensitive)', json.dumps({k: Q5[k]['native'] for k in Q5})),
    ('G4 edge enrichment interpretable', str(G4_PASS)),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 9 · Figures (plots only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 42 * MM), sharey=True)
for ax, run in zip(axes, RUNS):
    for spec in RUNS[run]:
        v = tot.loc[(run, spec)]
        ax.plot(v.index, v.to_numpy(), marker='o', ms=2, color=SPEC_COLOR[spec], label=spec)
    ax.set_xlabel('depth from the unit edge (2 µm bins)')
    ax.set_xticks(range(N_LAYERS + 1), [*map(str, range(N_LAYERS)), f'≥{N_LAYERS}'])
    ax.set_title({'aki': 'AKI run', 'hvm': 'human vs mouse run'}[run])
axes[0].set_ylabel('share of counts on non-native markers')
axes[1].legend(loc='upper right')
save(fig, '104a_depth_profile')
plt.show()

fig, axes = plt.subplots(2, len(CLASSES), figsize=(180 * MM, 60 * MM), sharex=True)
mids = (edges[:-1] + edges[1:]) / 2
for row, run in enumerate(RUNS):
    for col, k in enumerate(CLASSES):
        ax = axes[row, col]
        for spec in RUNS[run]:
            v = ALONG.loc[(run, spec), f'f {k}']
            ax.plot(mids[v.index.to_numpy()], v.to_numpy(), color=SPEC_COLOR[spec], lw=.8)
        if row == 0:
            ax.set_title(k)
        if row == 1:
            ax.set_xlabel('PT position')
    axes[row, 0].set_ylabel('f (mean)')
save(fig, '104b_fractions_along_axis')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 50 * MM))
for ax, (sp, pair) in zip(axes, (('mouse', MOUSE), ('human', HUMAN))):
    g = GRAD[GRAD.species == sp]
    full = g[[f'Ê full {s}' for s in pair]].mean(axis=1)
    core = g[[f'Ê core{CORE} {s}' for s in pair]].mean(axis=1)
    sens = g[f'sensitive core{CORE}'].to_numpy()
    ax.scatter(full[~sens], core[~sens], s=2, color=MUTED, lw=0)
    ax.scatter(full[sens], core[sens], s=5, color='#D55E00', lw=0)
    lim = float(np.nanmax(np.abs(np.r_[full, core]))) * 1.05
    ax.plot([-lim, lim], [-lim, lim], color=LIGHT, lw=.5)
    ax.axhline(0, color=LIGHT, lw=.5); ax.axvline(0, color=LIGHT, lw=.5)
    ax.set_xlabel('end contrast, full units'); ax.set_ylabel('end contrast, core bins')
    ax.set_title(sp)
save(fig, '104c_gradients_full_core')
plt.show()

fig, ax = plt.subplots(figsize=(80 * MM, 45 * MM))
lab = [f"{r.method} · {'AKI' if r.contrast.startswith('aki') else 'hvm'}" for r in EXPOSURE.itertuples()]
yy = np.arange(len(EXPOSURE))
ax.scatter(EXPOSURE.exposure, yy, color=INK, s=10, zorder=3)
ax.scatter(EXPOSURE['null mean'], yy, color=LIGHT, s=10, marker='s', zorder=2)
ax.set_yticks(yy, lab)
ax.set_xlabel('share of driver genes edge-enriched')
save(fig, '104d_method_exposure')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Human bins are matched to the export through its human_symbol column and the stage-01 unit objects\' unique var names, in the bin matrix\'s feature order.'),
    ('reading', 'The PT interior reference pools the core bins (depth ≥ 3) of the export PT units in the common support of each species.'),
    ('reading', 'Contact classes: PT, AL, DCT, CNT_CD, Glomerulus, outside, and other (SmoothMuscle, Unassigned, Rejected_ATL, unlabelled polygons).'),
    ('reading', 'Depth layers are floor(depth), with layers ≥ 6 pooled; the layers-≥-3 comparator is the bin-weighted mean of layers 3 to ≥ 6.'),
    ('correction', 'A scratch run first computed the Q2 depth reading on the fractions f, which subtract the pooled PT-interior share and so sit near 0 in the core by construction. The notebook computes it on the marker shares s_k, as the protocol states.'),
    ('post hoc', 'Mean F per specimen is added beside the frozen median: most units carry no counts on the few outside markers, so medians sit at the floor set by the pooled interior baseline.'),
    ('reading', 'The random aggregates are size-matched to the depth-≥-3 core only; at depth ≥ 4 and ≥ 5 the core holds fewer counts than R1 and R2, so the ladder\'s sensitive shares there include count loss.'),
    ('reading', 'Expression deciles for the Q6 null are from the mean over specimens of the pooled full-unit log2 CP10k in the run.'),
    ('reading', 'Q6 driver genes are the union of the driver gene lists of each method\'s calls in notebook 93\'s call tables, within notebook 71\'s universe.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '104_pt_contamination_audit', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'q1': Q1.round(4).reset_index().to_dict('records'), 'q2_depth': Q2_DEPTH.round(4).reset_index().to_dict('records'),
       'q2_contact': Q2_CONTACT.round(3).reset_index().to_dict('records'), 'q3_end': Q3_END.round(4).reset_index().to_dict('records'),
       'q3_aki': {'f_outside IR-control': IR_OUTSIDE, 'F IR-control': IR_F}, 'q4': Q4, 'q5': Q5,
       'g4': G4.round(3).to_dict('records'), 'exposure': EXPOSURE.drop(columns='flagged genes').round(4).to_dict('records'),
       'recovery': REC.to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('q4', 'q5', 'seconds')}, indent=2, default=str))